# Week 5 — Solved

Geospatial visualization: Choropleth maps with Plotly, interactive maps with Folium, and spatio-temporal analysis.

## Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, warnings, requests
warnings.filterwarnings('ignore')

# Load pre-merged dataset (from Week 2)
try:
    merged = pd.read_parquet('/home/claude/notebooks/sf_crime_merged.parquet')
    print(f'Loaded merged dataset: {len(merged):,} rows')
except FileNotFoundError:
    print('Run Week 2 first to generate the merged parquet, or re-fetch here.')
    # Fallback: re-fetch just 'now' data for standalone use
    import requests
    API_NOW = 'https://data.sfgov.org/resource/wg3w-h783.json'
    def fetch_all(url, params=None, chunk=50_000):
        if params is None: params = {}
        records, offset = [], 0
        while True:
            r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
            r.raise_for_status(); batch = r.json()
            if not batch: break
            records.extend(batch); offset += chunk
            if len(batch) < chunk: break
        return pd.DataFrame(records)
    raw = fetch_all(API_NOW)
    raw['date'] = pd.to_datetime(raw.get('incident_date', pd.NaT), errors='coerce')
    raw['year'] = raw['date'].dt.year
    raw['month'] = raw['date'].dt.month
    raw['weekday'] = raw['date'].dt.day_name()
    raw['hour'] = pd.to_datetime(raw.get('incident_time',''), format='%H:%M', errors='coerce').dt.hour
    raw['category'] = raw.get('incident_category', '')
    raw['district'] = raw.get('police_district', '')
    raw['lat'] = pd.to_numeric(raw.get('latitude', None), errors='coerce')
    raw['lon'] = pd.to_numeric(raw.get('longitude', None), errors='coerce')
    merged = raw.dropna(subset=['date']).copy()

FOCUS_CRIMES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
FOCUS_CRIMES = [c for c in FOCUS_CRIMES if c in merged['category'].unique()]
df_focus = merged[merged['category'].isin(FOCUS_CRIMES)].copy()
print(f'Focus crimes available: {len(FOCUS_CRIMES)}')


In [ ]:
# Additional imports for geo work
try:
    import plotly.express as px
    import plotly.graph_objects as go
    print('Plotly available')
except ImportError:
    print('Install: pip install plotly')

try:
    import folium
    from folium.plugins import HeatMap, HeatMapWithTime
    print('Folium available')
except ImportError:
    print('Install: pip install folium')


## Exercise 1.1 — Choropleth with random data

In [ ]:
import json, urllib.request

# Download SF police district GeoJSON
geojson_url = 'https://raw.githubusercontent.com/suneman/socialdata2025/main/files/sfpd.geojson'
try:
    with urllib.request.urlopen(geojson_url) as resp:
        sfpd_geojson = json.loads(resp.read())
    print('GeoJSON loaded, features:', len(sfpd_geojson['features']))
except Exception as e:
    print(f'Could not load GeoJSON: {e}')
    sfpd_geojson = None


In [ ]:
import random, plotly.express as px

random.seed(42)
districts = ['BAYVIEW','CENTRAL','INGLESIDE','MISSION','NORTHERN',
             'PARK','RICHMOND','SOUTHERN','TARAVAL','TENDERLOIN']
random_data = {d: random.random() for d in districts}
rand_df = pd.DataFrame({'district': list(random_data.keys()), 'value': list(random_data.values())})

if sfpd_geojson:
    fig = px.choropleth_mapbox(
        rand_df, geojson=sfpd_geojson,
        locations='district', featureidkey='properties.DISTRICT',
        color='value', color_continuous_scale='Viridis',
        range_color=(0, 1),
        mapbox_style='open-street-map',
        zoom=11, center={'lat': 37.7749, 'lon': -122.4194},
        opacity=0.6,
        title='SF Police Districts — Random Values'
    )
    fig.write_html('choropleth_random.html')
    fig.show()
    print('Saved choropleth_random.html')
else:
    print('Skipping choropleth (GeoJSON unavailable)')


## Exercise 1.2 — Choropleth with real crime ratios

In [ ]:
# Compute P(crime|district)/P(crime) for Larceny Theft
crime_sel = 'Larceny Theft'
p_crime = (df_focus['category'] == crime_sel).mean()
d_counts = df_focus.groupby('district').apply(
    lambda g: (g['category'] == crime_sel).mean()
)
ratio_series = (d_counts / p_crime).reset_index()
ratio_series.columns = ['district','ratio']
# Normalize district names to uppercase
ratio_series['district'] = ratio_series['district'].str.upper()

print(ratio_series.sort_values('ratio', ascending=False).to_string(index=False))

if sfpd_geojson:
    fig = px.choropleth_mapbox(
        ratio_series, geojson=sfpd_geojson,
        locations='district', featureidkey='properties.DISTRICT',
        color='ratio',
        color_continuous_scale='RdBu_r',
        color_continuous_midpoint=1.0,
        range_color=(0, 3),
        mapbox_style='open-street-map',
        zoom=11, center={'lat': 37.7749, 'lon': -122.4194},
        opacity=0.65,
        title=f'P({crime_sel}|district) / P({crime_sel}) — Geographic Ratio',
        labels={'ratio': 'Over/under index'}
    )
    fig.write_html('choropleth_larceny_ratio.html')
    fig.show()
    print('Saved choropleth_larceny_ratio.html')


**Comment:** The map makes geographic concentration immediately apparent — patterns that required careful reading of a bar chart become obvious spatially. The Tenderloin and Southern districts tend to be heavily over-represented for Larceny Theft (index > 2), consistent with high foot traffic in downtown retail and tourism corridors. Richmond and Taraval (residential, western neighborhoods) are under-represented.

## Exercise 1.3 — Where to park on Sundays (Motor Vehicle Theft by district)

In [ ]:
mvt_sunday = df_focus[
    (df_focus['category'] == 'Motor Vehicle Theft') &
    (df_focus['weekday'] == 'Sunday')
].groupby('district').size().reset_index(name='count')
mvt_sunday['district'] = mvt_sunday['district'].str.upper()

print('Motor Vehicle Theft on Sundays by district:')
print(mvt_sunday.sort_values('count').to_string(index=False))
print(f'\nSafest: {mvt_sunday.loc[mvt_sunday["count"].idxmin(), "district"]}')
print(f'Worst : {mvt_sunday.loc[mvt_sunday["count"].idxmax(), "district"]}')

if sfpd_geojson:
    fig = px.choropleth_mapbox(
        mvt_sunday, geojson=sfpd_geojson,
        locations='district', featureidkey='properties.DISTRICT',
        color='count', color_continuous_scale='Reds',
        mapbox_style='open-street-map',
        zoom=11, center={'lat': 37.7749, 'lon': -122.4194},
        opacity=0.65,
        title='Motor Vehicle Theft on Sundays by District',
        labels={'count': 'Thefts'}
    )
    fig.write_html('choropleth_mvt_sunday.html')
    fig.show()


## Exercise 2.1 — Folium point scatter map

In [ ]:
import folium

m = folium.Map([37.7749, -122.4194], zoom_start=13)
folium.Marker([37.77919, -122.41914],
              popup='SF City Hall',
              tooltip='SF City Hall',
              icon=folium.Icon(color='blue', icon='info-sign')).add_to(m)

# Drug Offense points — limited to 2 months to avoid memory issues
drug_2mo = df_focus[
    (df_focus['category'] == 'Drug Offense') &
    (df_focus['year'] == 2022) &
    (df_focus['month'].isin([6,7]))
].dropna(subset=['lat','lon'])
drug_2mo = drug_2mo[(drug_2mo['lat'].between(37.70, 37.83)) &
                    (drug_2mo['lon'].between(-122.53, -122.35))]

for _, row in drug_2mo.iterrows():
    folium.CircleMarker([row['lat'], row['lon']],
                        radius=3, color='red', fill=True,
                        fill_opacity=0.5).add_to(m)

m.save('scatter_drug.html')
print(f'Plotted {len(drug_2mo)} Drug Offense incidents. Saved scatter_drug.html')
m


## Exercise 2.2 — Heatmap

In [ ]:
from folium.plugins import HeatMap

# All-time Prostitution heatmap
prost = df_focus[(df_focus['category']=='Prostitution')].dropna(subset=['lat','lon'])
prost = prost[(prost['lat'].between(37.70, 37.83)) &
              (prost['lon'].between(-122.53, -122.35))]

m2 = folium.Map([37.7749, -122.4194], zoom_start=13)
HeatMap(prost[['lat','lon']].values.tolist(), radius=15, blur=10).add_to(m2)
m2.save('heatmap_prostitution.html')
print(f'Heatmap of {len(prost)} Prostitution incidents saved.')
m2


**Scatter vs Heatmap:**
- Scatter plots show *individual* incident locations and reveal exact hotspot streets. They can become unreadable with many points.
- Heatmaps show *density* — making overall concentration patterns immediately visible, but losing individual event precision. They're better for communicating 'where is crime concentrated' to a general audience.

## Exercise 2.3 — HeatMapWithTime animation

In [ ]:
from folium.plugins import HeatMapWithTime

# Larceny Theft monthly heatmap movie
crime_anim = 'Larceny Theft'
anim_data = df_focus[(df_focus['category']==crime_anim)].dropna(subset=['lat','lon'])
anim_data = anim_data[(anim_data['lat'].between(37.70, 37.83)) &
                      (anim_data['lon'].between(-122.53, -122.35))].copy()
anim_data['year_month'] = anim_data['year'].astype(str) + '-' + anim_data['month'].astype(str).str.zfill(2)

frames = []
labels = []
for ym in sorted(anim_data['year_month'].unique()):
    pts = anim_data[anim_data['year_month']==ym][['lat','lon']].values.tolist()
    frames.append(pts)
    labels.append(ym)

print(f'Animation: {len(frames)} frames ({labels[0]} → {labels[-1]})')

m3 = folium.Map([37.7749, -122.4194], zoom_start=13)
HeatMapWithTime(frames, index=labels, auto_play=False,
                min_opacity=0.3, radius=12).add_to(m3)
m3.save('heatmap_movie_larceny.html')
print('Saved heatmap_movie_larceny.html')


## Exercise 3.1 — Spatio-temporal analysis of Prostitution

In [ ]:
# Full spatio-temporal analysis of Prostitution in SF
prost = df_focus[df_focus['category']=='Prostitution'].copy()
prost = prost.dropna(subset=['lat','lon','year','hour'])
prost = prost[(prost['lat'].between(37.70, 37.83)) &
              (prost['lon'].between(-122.53, -122.35))]

# 1. Yearly trend
yearly_p = prost.groupby('year').size()
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(yearly_p.index, yearly_p.values, color='purple', alpha=0.8)
ax.set_title('Prostitution Incidents per Year', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Incidents')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('prost_yearly.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# 2. Hour-of-day distribution
hour_p = prost.groupby('hour').size()
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(hour_p.index, hour_p.values, color='purple', alpha=0.8)
ax.set_title('Prostitution by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour'); ax.set_ylabel('Incidents'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('prost_hourly.png', dpi=150, bbox_inches='tight'); plt.show()
print('Strong overnight concentration (11pm–2am) is clearly visible.')


In [ ]:
# 3. District breakdown
dist_p = prost.groupby('district').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(dist_p.index, dist_p.values, color='purple', alpha=0.8)
ax.set_title('Prostitution by Police District', fontweight='bold')
ax.set_xticklabels(dist_p.index, rotation=45, ha='right')
ax.set_ylabel('Incidents'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('prost_district.png', dpi=150, bbox_inches='tight'); plt.show()


**Analysis of Prostitution patterns:**

Prostitution in SF shows extreme spatio-temporal concentration: overwhelmingly in Mission and Tenderloin districts, and almost exclusively between 10pm and 3am. The sharp decline from ~2018 onward likely reflects FOSTA-SESTA legislation (2018) which moved outdoor sex work off the streets by disrupting online advertising, rather than a reduction in sex work itself — another example of a policy change masquerading as a crime change in the data.

**Role of data errors:** GPS coordinates for incidents recorded by police are often approximate (recorded at the nearest intersection or at the patrol car's location) rather than the exact incident location. This means hotspot maps overstate concentration at specific intersections.

**Feedback loop concern:** If this hotspot map were used to deploy more police to Mission and Tenderloin, arrests would increase in those areas, the data would show more crime there, justifying even more deployment — regardless of whether the underlying behavior actually changed.